# 05 Image Caption and Emotion

## Goal
This notebook generates image-derived data from the downloaded image dataset.

The notebook will:
- load the image scraping metadata
- caption downloaded images using an image captioning model
- assign simple emotion labels to captions
- export the results for later multimodal comparison

This notebook builds the image-derived branch of the project pipeline.

In [2]:
# 安装依赖（只需要跑一次）
!pip install --upgrade pip
!pip install transformers accelerate sentencepiece
!pip install torch torchvision --index-url https://download.pytorch.org/whl/cu121
!pip install pillow

  Using cached transformers-5.5.4-py3-none-any.whl.metadata (32 kB)
  Using cached tokenizers-0.22.2-cp39-abi3-win_amd64.whl.metadata (7.4 kB)
Using cached transformers-5.5.4-py3-none-any.whl (10.2 MB)
Using cached tokenizers-0.22.2-cp39-abi3-win_amd64.whl (2.7 MB)
   ---------------------------------------- 0.0/1.1 MB ? eta -:--:--
   ---------------------------------------- 1.1/1.1 MB 25.6 MB/s  0:00:00

   -------------------- ------------------- 2/4 [accelerate]
   -------------------- ------------------- 2/4 [accelerate]
   ------------------------------ --------- 3/4 [transformers]
   ------------------------------ --------- 3/4 [transformers]
   ------------------------------ --------- 3/4 [transformers]
   ------------------------------ --------- 3/4 [transformers]
   ------------------------------ --------- 3/4 [transformers]
   ------------------------------ --------- 3/4 [transformers]
   ------------------------------ --------- 3/4 [transformers]
   ------------------------

In [3]:
import torch
print("Torch version:", torch.__version__)
print("CUDA available:", torch.cuda.is_available())
print("GPU:", torch.cuda.get_device_name(0) if torch.cuda.is_available() else "CPU")

Torch version: 2.12.0.dev20260408+cu128
CUDA available: True
GPU: NVIDIA GeForce RTX 5090


## Step 1 — Import libraries
Import the libraries needed for metadata loading, image processing, caption generation, and dataframe handling.

In [4]:
import sys
!{sys.executable} -m pip install transformers accelerate sentencepiece

In [5]:
from pathlib import Path
import json
import re
import pandas as pd
from PIL import Image

import torch
from transformers import BlipProcessor, BlipForConditionalGeneration

c:\Users\Greeny\anaconda3\envs\surface\lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


## Step 2 — Load the project config
Load the shared project paths created earlier.

In [9]:
PROJECT_ROOT = Path.cwd().resolve().parent
CONFIG_PATH = PROJECT_ROOT / "src" / "project_config.json"

with open(CONFIG_PATH, "r", encoding="utf-8") as f:
    config = json.load(f)

OUTPUT_TABLES_DIR = Path(config["output_tables_dir"])
RAW_IMAGES_DIR = Path(config["raw_images_dir"])

if "processed_image_derived_dir" in config:
    PROCESSED_IMAGE_DERIVED_DIR = Path(config["processed_image_derived_dir"])
else:
    PROCESSED_IMAGE_DERIVED_DIR = PROJECT_ROOT / "data" / "processed" / "image_derived_data"
    PROCESSED_IMAGE_DERIVED_DIR.mkdir(parents=True, exist_ok=True)

print("Output tables dir:", OUTPUT_TABLES_DIR)
print("Raw images dir:", RAW_IMAGES_DIR)
print("Processed image-derived dir:", PROCESSED_IMAGE_DERIVED_DIR)

Output tables dir: D:\Work\Workspace\Projects\Python\data-driven-surface\outputs\tables
Raw images dir: D:\Work\Workspace\Projects\Python\data-driven-surface\data\raw\scraped_images
Processed image-derived dir: D:\Work\Workspace\Projects\Python\data-driven-surface\data\processed\image_derived_data


## Step 3 — Load image inspection data
Read the image inspection table created in `03b_image_dataset_inspection.ipynb`.

In [10]:
inspection_csv_path = OUTPUT_TABLES_DIR / "image_dataset_inspection.csv"

if not inspection_csv_path.exists():
    raise FileNotFoundError(
        f"Inspection CSV not found: {inspection_csv_path}\n"
        "Please run 03b_image_dataset_inspection.ipynb first."
    )

image_df = pd.read_csv(inspection_csv_path)
image_df.head()

,unit_id,city_group,search_query,image_rank,image_url,source_url,title,local_path,download_success,error_x,file_exists,width,height,valid_image,error_y
0,1,Cities & Memory 1,"surreal city architecture, silver, domes, bron...",1,https://pics.craiyon.com/2023-10-24/332aa5adef...,https://www.craiyon.com/image/BtAnG3xCRzWOAD7q...,Cityscape with silver domes and bronze statues...,D:\Work\Workspace\Projects\Python\data-driven-...,True,NaN,True,1024.0,1024.0,True,NaN
1,1,Cities & Memory 1,"surreal city architecture, silver, domes, bron...",2,https://pics.craiyon.com/2024-04-28/oTRPsNoNT5...,https://www.craiyon.com/image/Vmzk7YtkTdWm6uDu...,"City with silver domes, bronze statues, paved ...",D:\Work\Workspace\Projects\Python\data-driven-...,True,NaN,True,1024.0,1024.0,True,NaN
2,1,Cities & Memory 1,"surreal city architecture, silver, domes, bron...",3,https://pics.craiyon.com/2023-10-07/d79e0b758b...,https://www.craiyon.com/search/aerial+view+of+...,aerial view of a city with silver domes and br...,D:\Work\Workspace\Projects\Python\data-driven-...,True,NaN,True,1024.0,1024.0,True,NaN
3,2,Cities & Memory 2,"surreal city architecture, spiral, buildings, ...",1,https://images.stockcake.com/public/8/e/d/8ed0...,https://stockcake.com/i/spiral-urban-dreamscap...,"Free Spiral Urban Dreamscape Image - Spiral, S...",D:\Work\Workspace\Projects\Python\data-driven-...,True,NaN,True,384.0,672.0,True,NaN
4,2,Cities & Memory 2,"surreal city architecture, spiral, buildings, ...",2,https://cdn.openart.ai/published/uNXcBp8dNKILa...,https://openart.ai/search/architecture,OpenArt - Find and Easily Create Customized ar...,D:\Work\Workspace\Projects\Python\data-driven-...,True,NaN,True,512.0,819.0,True,NaN


## Step 4 — Keep only valid images
Filter the dataset so that captioning only runs on valid image files.

In [11]:
valid_image_df = image_df[image_df["valid_image"] == True].copy()
valid_image_df = valid_image_df.reset_index(drop=True)

print("Number of valid images:", len(valid_image_df))
valid_image_df.head()

Number of valid images: 132


,unit_id,city_group,search_query,image_rank,image_url,source_url,title,local_path,download_success,error_x,file_exists,width,height,valid_image,error_y
0,1,Cities & Memory 1,"surreal city architecture, silver, domes, bron...",1,https://pics.craiyon.com/2023-10-24/332aa5adef...,https://www.craiyon.com/image/BtAnG3xCRzWOAD7q...,Cityscape with silver domes and bronze statues...,D:\Work\Workspace\Projects\Python\data-driven-...,True,NaN,True,1024.0,1024.0,True,NaN
1,1,Cities & Memory 1,"surreal city architecture, silver, domes, bron...",2,https://pics.craiyon.com/2024-04-28/oTRPsNoNT5...,https://www.craiyon.com/image/Vmzk7YtkTdWm6uDu...,"City with silver domes, bronze statues, paved ...",D:\Work\Workspace\Projects\Python\data-driven-...,True,NaN,True,1024.0,1024.0,True,NaN
2,1,Cities & Memory 1,"surreal city architecture, silver, domes, bron...",3,https://pics.craiyon.com/2023-10-07/d79e0b758b...,https://www.craiyon.com/search/aerial+view+of+...,aerial view of a city with silver domes and br...,D:\Work\Workspace\Projects\Python\data-driven-...,True,NaN,True,1024.0,1024.0,True,NaN
3,2,Cities & Memory 2,"surreal city architecture, spiral, buildings, ...",1,https://images.stockcake.com/public/8/e/d/8ed0...,https://stockcake.com/i/spiral-urban-dreamscap...,"Free Spiral Urban Dreamscape Image - Spiral, S...",D:\Work\Workspace\Projects\Python\data-driven-...,True,NaN,True,384.0,672.0,True,NaN
4,2,Cities & Memory 2,"surreal city architecture, spiral, buildings, ...",2,https://cdn.openart.ai/published/uNXcBp8dNKILa...,https://openart.ai/search/architecture,OpenArt - Find and Easily Create Customized ar...,D:\Work\Workspace\Projects\Python\data-driven-...,True,NaN,True,512.0,819.0,True,NaN


## Step 5 — Load the BLIP captioning model
This step loads a pre-trained image captioning model from Hugging Face.

In [12]:
device = "cuda" if torch.cuda.is_available() else "cpu"
print("Using device:", device)

processor = BlipProcessor.from_pretrained("Salesforce/blip-image-captioning-base")
model = BlipForConditionalGeneration.from_pretrained("Salesforce/blip-image-captioning-base").to(device)

Using device: cuda


c:\Users\Greeny\anaconda3\envs\surface\lib\site-packages\huggingface_hub\file_download.py:138: UserWarning: `huggingface_hub` cache-system uses symlinks by default to efficiently store duplicated files but your machine does not support them in C:\Users\Greeny\.cache\huggingface\hub\models--Salesforce--blip-image-captioning-base. Caching files will still work but in a degraded version that might require more space on your disk. This warning can be disabled by setting the `HF_HUB_DISABLE_SYMLINKS_WARNING` environment variable. For more details, see https://huggingface.co/docs/huggingface_hub/how-to-cache#limitations.
To support symlinks on Windows, you either need to activate Developer Mode or to run Python as an administrator. In order to activate developer mode, see this article: https://docs.microsoft.com/en-us/windows/apps/get-started/enable-your-device-for-development
  warnings.warn(message)
Loading weights: 100%|██████████| 473/473 [00:00<00:00, 46651.60it/s]
BlipForConditionalGen

## Step 6 — Define an image captioning helper
This function opens an image, runs caption generation, and returns a text description.

In [15]:
def generate_caption(image_path: str, max_new_tokens: int = 30):
    image = Image.open(image_path).convert("RGB")

    inputs = processor(images=image, return_tensors="pt").to(device)
    output = model.generate(**inputs, max_new_tokens=max_new_tokens)

    caption = processor.decode(output[0], skip_special_tokens=True)
    return caption.strip()

In [19]:
def enhance_caption(caption):
    caption = caption.lower().strip()

    if any(word in caption for word in ["desert", "sand"]):
        if caption.startswith("a "):
            caption = "a surreal desert scene with " + caption[2:]
        else:
            caption = "a surreal desert scene with " + caption

    elif any(word in caption for word in ["city", "building", "street", "tower", "staircase", "castle"]):
        if caption.startswith("a "):
            caption = "a surreal " + caption[2:]
        else:
            caption = "a surreal " + caption

    if any(word in caption for word in ["night", "dark"]):
        caption += " at night"

    return caption

## Step 7 — Test caption generation on a small subset
Generate captions for a few sample images first before processing the full dataset.

In [20]:
test_caption_df = valid_image_df.head(5).copy()

test_captions = []
for row in test_caption_df.itertuples(index=False):
    caption = generate_caption(row.local_path)
    caption = enhance_caption(caption)
    test_captions.append(caption)
    print(f"Unit {row.unit_id} | Rank {row.image_rank}")
    print("Caption:", caption)
    print("-" * 80)

test_caption_df["caption"] = test_captions
test_caption_df[["unit_id", "city_group", "image_rank", "caption"]]

Unit 1 | Rank 1
Caption: a surreal castle with a clock in the middle
--------------------------------------------------------------------------------
Unit 1 | Rank 2
Caption: a large fountain with statues on it
--------------------------------------------------------------------------------
Unit 1 | Rank 3
Caption: a surreal circular view of a city with buildings
--------------------------------------------------------------------------------
Unit 2 | Rank 1
Caption: a surreal spiral staircase in a city with a pink light
--------------------------------------------------------------------------------
Unit 2 | Rank 2
Caption: a surreal city with a staircase leading to a building
--------------------------------------------------------------------------------


,unit_id,city_group,image_rank,caption
0,1,Cities & Memory 1,1,a surreal castle with a clock in the middle
1,1,Cities & Memory 1,2,a large fountain with statues on it
2,1,Cities & Memory 1,3,a surreal circular view of a city with buildings
3,2,Cities & Memory 2,1,a surreal spiral staircase in a city with a pi...
4,2,Cities & Memory 2,2,a surreal city with a staircase leading to a b...


## Step 8 — Define a simple caption-based emotion function
This provides a lightweight baseline for identifying emotional tone in generated captions.

In [25]:
caption_emotion_lexicon = {
    "joy": [
        "bright", "colorful", "sunny", "happy", "light", "warm"
    ],
    "sadness": [
        "empty", "abandoned", "lonely", "ruins", "decay"
    ],
    "fear": [
        "dark", "shadow", "ominous", "danger", "void"
    ],
    "desire": [
        "dream", "surreal", "fantasy", "mystical", "floating", "impossible"
    ],
    "awe": [
        "grand", "vast", "towering", "massive", "ornate",
        "cathedral", "palace", "monumental", "huge", "epic"
    ]
}

def tokenize_caption(text: str):
    return re.findall(r"\b[a-zA-Z']+\b", text.lower())

def score_caption_emotions(text: str, lexicon: dict):
    text = text.lower()

    # 👇 新增规则（关键）
    if any(word in text for word in ["castle", "tower", "palace"]):
        return {"awe": 2}
    
    if any(word in text for word in ["staircase", "path", "leading"]):
        return {"desire": 2}

    tokens = tokenize_caption(text)
    scores = {}
    for emotion, vocab in lexicon.items():
        scores[emotion] = sum(1 for token in tokens if token in vocab)
    return scores

def get_dominant_caption_emotion(score_dict: dict):
    if not score_dict:
        return "neutral"

    max_score = max(score_dict.values())
    if max_score == 0:
        return "neutral"

    top = [k for k, v in score_dict.items() if v == max_score]
    return top[0]

## Step 9 — Test emotion scoring on sample captions
Apply the baseline emotion analysis to the test subset.

In [26]:
test_caption_df["caption_emotion_scores"] = test_caption_df["caption"].apply(
    lambda x: score_caption_emotions(x, caption_emotion_lexicon)
)
test_caption_df["caption_dominant_emotion"] = test_caption_df["caption_emotion_scores"].apply(
    get_dominant_caption_emotion
)

test_caption_df[[
    "unit_id",
    "city_group",
    "image_rank",
    "caption",
    "caption_dominant_emotion"
]]

,unit_id,city_group,image_rank,caption,caption_dominant_emotion
0,1,Cities & Memory 1,1,a surreal castle with a clock in the middle,awe
1,1,Cities & Memory 1,2,a large fountain with statues on it,neutral
2,1,Cities & Memory 1,3,a surreal circular view of a city with buildings,desire
3,2,Cities & Memory 2,1,a surreal spiral staircase in a city with a pi...,desire
4,2,Cities & Memory 2,2,a surreal city with a staircase leading to a b...,desire


## Step 10 — Run caption generation on the full valid image dataset
Once the test results look reasonable, generate captions for all valid images.

In [27]:
run_full_captioning = True

In [28]:
if run_full_captioning:
    captions = []

    for i, row in enumerate(valid_image_df.itertuples(index=False), start=1):
        caption = generate_caption(row.local_path)
        captions.append(caption)

        if i % 20 == 0 or i == len(valid_image_df):
            print(f"Processed {i}/{len(valid_image_df)} images")

    valid_image_df["caption"] = captions
else:
    print("Full captioning is disabled. Set run_full_captioning = True to run it.")

Processed 20/132 images
Processed 40/132 images
Processed 60/132 images
Processed 80/132 images
Processed 100/132 images
Processed 120/132 images
Processed 132/132 images


## Step 11 — Apply caption-based emotion analysis to the full dataset
Assign baseline emotion labels to all generated captions.

In [29]:
if "caption" in valid_image_df.columns:
    valid_image_df["caption_emotion_scores"] = valid_image_df["caption"].apply(
        lambda x: score_caption_emotions(x, caption_emotion_lexicon)
    )
    valid_image_df["caption_dominant_emotion"] = valid_image_df["caption_emotion_scores"].apply(
        get_dominant_caption_emotion
    )

    emotion_score_df = pd.json_normalize(valid_image_df["caption_emotion_scores"])
    valid_image_df = pd.concat(
        [valid_image_df.drop(columns=["caption_emotion_scores"]), emotion_score_df],
        axis=1
    )

valid_image_df.head()

,unit_id,city_group,search_query,image_rank,image_url,source_url,title,local_path,download_success,error_x,...,height,valid_image,error_y,caption,caption_dominant_emotion,awe,joy,sadness,fear,desire
0,1,Cities & Memory 1,"surreal city architecture, silver, domes, bron...",1,https://pics.craiyon.com/2023-10-24/332aa5adef...,https://www.craiyon.com/image/BtAnG3xCRzWOAD7q...,Cityscape with silver domes and bronze statues...,D:\Work\Workspace\Projects\Python\data-driven-...,True,NaN,...,1024.0,True,NaN,a castle with a clock in the middle,awe,2.0,NaN,NaN,NaN,NaN
1,1,Cities & Memory 1,"surreal city architecture, silver, domes, bron...",2,https://pics.craiyon.com/2024-04-28/oTRPsNoNT5...,https://www.craiyon.com/image/Vmzk7YtkTdWm6uDu...,"City with silver domes, bronze statues, paved ...",D:\Work\Workspace\Projects\Python\data-driven-...,True,NaN,...,1024.0,True,NaN,a large fountain with statues on it,neutral,0.0,0.0,0.0,0.0,0.0
2,1,Cities & Memory 1,"surreal city architecture, silver, domes, bron...",3,https://pics.craiyon.com/2023-10-07/d79e0b758b...,https://www.craiyon.com/search/aerial+view+of+...,aerial view of a city with silver domes and br...,D:\Work\Workspace\Projects\Python\data-driven-...,True,NaN,...,1024.0,True,NaN,a circular view of a city with buildings,neutral,0.0,0.0,0.0,0.0,0.0
3,2,Cities & Memory 2,"surreal city architecture, spiral, buildings, ...",1,https://images.stockcake.com/public/8/e/d/8ed0...,https://stockcake.com/i/spiral-urban-dreamscap...,"Free Spiral Urban Dreamscape Image - Spiral, S...",D:\Work\Workspace\Projects\Python\data-driven-...,True,NaN,...,672.0,True,NaN,a spiral staircase in a city with a pink light,desire,NaN,NaN,NaN,NaN,2.0
4,2,Cities & Memory 2,"surreal city architecture, spiral, buildings, ...",2,https://cdn.openart.ai/published/uNXcBp8dNKILa...,https://openart.ai/search/architecture,OpenArt - Find and Easily Create Customized ar...,D:\Work\Workspace\Projects\Python\data-driven-...,True,NaN,...,819.0,True,NaN,a city with a staircase leading to a building,desire,NaN,NaN,NaN,NaN,2.0


## Step 12 — Inspect sample caption results
Preview the generated captions and emotion labels.

In [30]:
valid_image_df[[
    "unit_id",
    "city_group",
    "image_rank",
    "caption",
    "caption_dominant_emotion"
]].head(20)

,unit_id,city_group,image_rank,caption,caption_dominant_emotion
0,1,Cities & Memory 1,1,a castle with a clock in the middle,awe
1,1,Cities & Memory 1,2,a large fountain with statues on it,neutral
2,1,Cities & Memory 1,3,a circular view of a city with buildings,neutral
3,2,Cities & Memory 2,1,a spiral staircase in a city with a pink light,desire
4,2,Cities & Memory 2,2,a city with a staircase leading to a building,desire
5,2,Cities & Memory 2,3,a painting of a city with a red roof,neutral
6,3,Cities & Desire 1,1,a desert scene with a desert city in the backg...,neutral
7,3,Cities & Desire 1,2,a desert with a path leading to a pyramid,desire
8,3,Cities & Desire 1,3,a desert with a large dome and several people ...,neutral
9,4,Cities & Memory 3,1,a surreal city with stairs leading to the sky,desire


## Step 13 — Save the caption dataset as CSV
Export the image-derived caption data for later multimodal comparison.

In [31]:
caption_csv_path = PROCESSED_IMAGE_DERIVED_DIR / "image_captions_processed.csv"
valid_image_df.to_csv(caption_csv_path, index=False, encoding="utf-8-sig")

print("Saved caption CSV to:", caption_csv_path)

Saved caption CSV to: D:\Work\Workspace\Projects\Python\data-driven-surface\data\processed\image_derived_data\image_captions_processed.csv


## Step 14 — Save the caption dataset as JSON
This JSON file is useful for cross mapping and later prompt construction.

In [32]:
caption_json_path = PROCESSED_IMAGE_DERIVED_DIR / "image_captions_processed.json"

caption_records = valid_image_df.to_dict(orient="records")

with open(caption_json_path, "w", encoding="utf-8") as f:
    json.dump(caption_records, f, indent=4, ensure_ascii=False)

print("Saved caption JSON to:", caption_json_path)

Saved caption JSON to: D:\Work\Workspace\Projects\Python\data-driven-surface\data\processed\image_derived_data\image_captions_processed.json


## Step 15 — Summary
This notebook has:
- loaded the valid scraped image dataset
- generated captions for downloaded images
- assigned baseline emotion labels to captions
- exported image-derived data for later multimodal comparison

Next notebook:
- `04_ai_text_generation.ipynb`